# (final) Expert evaluation framework

Prepares the **semantician evaluation**: our tokenizer vs the strongest baseline, judged against the official SNOMED CT definition.

Steps:

1. HERO concepts $M$ (same set as `(final)ehr_application_analysis.ipynb`)
2. Methods to compare: ours ($\lambda$ = 0.9, $k$ = 5,000) vs Personalized PageRank ($k$ = 5,000)
3. Tokenization of $M$ with every method
4. Comparison of the context trees
5. Number of concepts with the **same** tree
6. Concepts with a **different** tree, split into categories
7. Eligible pool (official concepts, short representations)
8. Sample by category and status, and their context trees
9. One rater form for every expert, and one ground-truth file

**Inclusion criteria for a concept** (all must hold):

| # | Criterion | Why |
|---|---|---|
| 1 | In $M$ (HERO concept in $G_D$) | same concept set as the paper |
| 2 | In the official SNOMED CT release (fully defined **or primitive**) | raters need the official definition; primitive concepts are a real part of SNOMED CT (88% of the official-release events), so leaving them out would bias the result |
| 3 | Context tree of ours ≠ context tree of the baseline | identical outputs need no rater: counted as "no difference" |
| 4 | At most `max_tokens_shown` tokens in A + B together | rater workload (see the note in Section 6–7) |

**Categories** (from the co-author's proposal). A concept can be kept **as a token** (the tree is the concept itself, no loss) or **decomposed** into other tokens:

| Category | Ours | Baseline | Raters? |
|---|---|---|---|
| `both_token` | token | token | no: identical, "no difference" |
| `same_tree` | decomposed | decomposed, same tree | no: identical, "no difference" |
| `ours_token` | token | decomposed | yes: does the baseline's decomposition lose meaning? |
| `baseline_token` | decomposed | token | yes: does our decomposition lose meaning? |
| `both_decompose` | decomposed | decomposed, different tree | yes: which decomposition is better? |

The rater task is the same everywhere: A better / no clear difference / B better, plus the problem checkboxes.

**Sample: stratified by category and status.** Each rated category gets a fixed number of concepts, split between fully defined and primitive in the same ratio as in the population of that category (the size limit removes more fully defined trees, so a plain random draw would skew the mix). Results are reported per stratum and weighted by the stratum sizes.

Inputs are the caches written by `(final)ehr_application_analysis.ipynb` (graph $G_D$, token rankings, result tables). Nothing is recomputed there.

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import json
import os
import pickle
from collections import Counter
from dataclasses import dataclass

import numpy as np
import polars as pl

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.fidelity_eval as fe
from src.graph_tokenizer_gd_tree_dev.eval import build_context_trees, _iter_contexts
from src.graph_tokenizer_gd_tree_dev.tokenizer import signature
from src.graph_tokenizer_gd_tree_dev.drilldown import render_context_tree

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(70);

## 0. Configuration

In [3]:
@dataclass(frozen=True)
class EvalConfig:
    # --- inputs ---------------------------------------------------------------------------------
    final_dir: str = "D:/greedy_graph_data/final_hero/"            # outputs of (final)ehr_application_analysis.ipynb
    vocab_path: str = config.HERODataVocabsTest.vocab_count_volumn  # HERO event counts per code
    concept_path: str = config.BasicConfig.concept_path             # id, label, concept_type (SCT_PRE / SCT_POST)
    release_path: str = config.BasicConfig.official_release_path    # official release: id, status (primitive / defined)
    data_max_year: int = 2025                                       # same filter as the final notebook

    # --- methods ----------------------------------------------------------------------------------
    D: int = config.TokenizerParam.max_dist_candidate               # search horizon (9)
    lam: str = "0.9"                                                # our configuration (file ours/<lam>.parquet)
    k: int = 5000                                                   # token budget of every method
    # baselines that can be compared (ablation and random are not baselines here)
    baselines: tuple = ("highest_degree_dist_1", "personalized_pagerank", "highest_degree",
                        "closeness_centrality", "pagerank", "most_children")
    # the baseline the raters compare with: best baseline of the paper's ranking table
    # (compression level that contains k)
    baseline: str = "personalized_pagerank"

    # --- sample -------------------------------------------------------------------------------------
    # concepts per rated category (each expert rates all of them); split by status within each category
    n_rated: tuple = (("ours_token", 30), ("baseline_token", 20), ("both_decompose", 50))
    max_tokens_shown: int = 16                                      # criterion 6: tokens in A + B together (rater workload)
    seed: int = 2026

    # --- outputs ----------------------------------------------------------------------------------
    out_dir: str = "D:/greedy_graph_data/final_hero/expert_eval/"


CFG = EvalConfig()
D = CFG.D
os.makedirs(CFG.out_dir, exist_ok=True)


def out(name):
    return os.path.join(CFG.out_dir, name)


N_RATED = dict(CFG.n_rated)
RATED = list(N_RATED)                                               # categories shown to raters
STATUSES = ["defined", "primitive"]                                 # official definition status
print(f"lambda = {CFG.lam}, k = {CFG.k:,}, D = {D}, seed = {CFG.seed}")
print(f"sample per expert: {N_RATED} ({sum(N_RATED.values())} concepts)")

lambda = 0.9, k = 5,000, D = 9, seed = 2026
sample per expert: {'ours_token': 30, 'baseline_token': 20, 'both_decompose': 50} (100 concepts)


## 1. HERO concepts $M$

Same definition as Section 1–2 of the final notebook: SNOMED CT codes of the HERO events (up to `data_max_year`) that are nodes of $G_D$.

In [4]:
with open(os.path.join(CFG.final_dir, "subgraph", "combined_subgraphs.pickle"), "rb") as f:
    G_D = pickle.load(f)

df_vocab = (pl.read_parquet(CFG.vocab_path)
              .filter(pl.col("date").dt.year() <= CFG.data_max_year)
              .filter(pl.col("coding_system") == "SNOMED"))
codes_data = sorted(df_vocab["code"].unique().to_list())
nodes_GD = set(G_D.nodes())
M = sorted(c for c in codes_data if c in nodes_GD)

df_cpt = pl.read_parquet(CFG.concept_path).select("id", "label", "concept_type")
id_to_label = dict(zip(df_cpt["id"], df_cpt["label"]))
concept_type = dict(zip(df_cpt["id"], df_cpt["concept_type"]))
label = lambda c: id_to_label.get(c, c)

# official status: one status per concept id (FSN and synonyms share it)
df_status = pl.read_parquet(CFG.release_path, columns=["id", "status"]).unique()
assert df_status["id"].n_unique() == df_status.height, "a concept has two statuses"
status = dict(df_status.iter_rows())

with open(os.path.join(CFG.final_dir, "tables", "setup.json")) as f:
    m_final = json.load(f)["|M| (codes in G_D)"]
assert len(M) == m_final, f"|M| = {len(M)} differs from the final notebook ({m_final})"

print(f"|M| = {len(M):,} (same as the final notebook)")
print("by concept type:", dict(Counter(concept_type.get(c, "unknown") for c in M)))
print("by official status:", dict(Counter(status.get(c, "not in release") for c in M)))

|M| = 16,370 (same as the final notebook)
by concept type: {'SCT_PRE': 11796, 'SCT_POST': 4574}
by official status: {'defined': 6845, 'primitive': 4951, 'not in release': 4574}


## 2. Methods to compare

Ours: $\lambda$ = `CFG.lam`, $k$ = `CFG.k`. For the baseline, two rankings are possible:

* **at exactly $k$** = `CFG.k` (average rank over the 4 core metrics, all 19 configurations);
* **over the compression level** that contains $k$ (the paper's ranking table, 9 budgets per level).

Both are printed for transparency. **Decision:** `CFG.baseline` = Personalized PageRank, the best baseline of the paper's own ranking table (compression level that contains $k$).

In [5]:
CORE, HIB = fe.CORE_METRICS, fe.HIGHER_IS_BETTER
NOT_BASELINE = {"ours", "ours(ablation)", "random"}

perf = pl.read_csv(os.path.join(CFG.final_dir, "results", "performance_all.csv"),
                   schema_overrides={"method": pl.String})
rank_k = (perf.filter(pl.col("k") == CFG.k)
              .with_columns([pl.col(m).rank("average", descending=HIB[m]).alias(f"{m}_rank") for m in CORE])
              .with_columns(pl.mean_horizontal([f"{m}_rank" for m in CORE]).alias("avg_rank"))
              .sort("avg_rank")
              .select("method", "family", *CORE, "avg_rank"))

rank_lvl = pl.read_csv(os.path.join(CFG.final_dir, "tables", "ranking.csv"), schema_overrides={"method": pl.String})
ks_bench = sorted(perf["k"].unique().to_list())
n_lvl = len(ks_bench) // 3
levels = {"high": ks_bench[:n_lvl], "mid": ks_bench[n_lvl:2 * n_lvl], "low": ks_bench[2 * n_lvl:]}
level_k = next(l for l, ks in levels.items() if CFG.k in ks)
rank_lvl = rank_lvl.filter(pl.col("level") == level_k).sort("avg_rank").select("method", "family", "avg_rank")

best_at_k = rank_k.filter(~pl.col("family").is_in(NOT_BASELINE))["method"][0]
best_lvl = rank_lvl.filter(~pl.col("family").is_in(NOT_BASELINE))["method"][0]
COMPARE = [CFG.baseline]                         # decision (see the markdown above)

print(f"best baseline at k = {CFG.k:,}: {best_at_k}")
print(f"best baseline over the '{level_k}' compression level (k = {levels[level_k][0]:,}-{levels[level_k][-1]:,}): {best_lvl}")
print("COMPARE =", COMPARE)
rank_k

best baseline at k = 5,000: highest_degree_dist_1
best baseline over the 'low' compression level (k = 4,750-6,750): personalized_pagerank
COMPARE = ['personalized_pagerank']


method,family,semantic_fidelity,unique_rate,unknown_rate,tokens_per_concept,avg_rank
str,str,f64,f64,f64,f64,f64
"""0.9""","""ours""",0.832405,0.790531,0.0,7.501222,4.25
"""0.8""","""ours""",0.82851,0.790104,0.0,7.630605,5.0
"""0.7""","""ours""",0.823307,0.78931,0.0,7.756261,6.0
"""0.6""","""ours""",0.815662,0.793769,0.0,7.981368,6.25
"""0.5""","""ours""",0.804898,0.79493,0.0,8.296029,7.75
"""highest_degree_dist_1""","""degree-based""",0.820661,0.755345,0.001588,7.418509,8.5
"""0.4""","""ours""",0.788277,0.794869,0.0,8.457911,8.75
"""0.1""","""ours""",0.693313,0.737569,0.0,5.205254,9.25
"""0.2""","""ours""",0.738692,0.767196,0.0,7.636775,9.5


In [6]:
rank_lvl

method,family,avg_rank
str,str,f64
"""0.9""","""ours""",4.638889
"""0.8""","""ours""",5.194444
"""0.7""","""ours""",5.416667
"""0.6""","""ours""",5.708333
"""0.5""","""ours""",7.319444
"""personalized_pagerank""","""centrality-based""",8.472222
"""0.4""","""ours""",8.791667
"""0.2""","""ours""",9.291667
"""0.3""","""ours""",9.319444


## 3. Tokenization

Context trees of all $|M|$ concepts for ours and every deterministic baseline, at the same budget $k$.

In [7]:
adj = tokenizer.build_out_adjacency(G_D)


def ranked_tokens(method):
    sub = "ours" if method == CFG.lam else "baselines"
    return pl.read_parquet(os.path.join(CFG.final_dir, sub, f"{method}.parquet"))["token"].to_list()


OURS = CFG.lam
METHODS = [OURS] + list(CFG.baselines)
T = {m: set(ranked_tokens(m)[:CFG.k]) for m in METHODS}
assert all(len(t) == CFG.k for t in T.values())

trees = {m: build_context_trees(M, adj, T[m], D, id_to_label) for m in METHODS}
sig = {m: {c: signature(tr) for c, tr in trees[m].items()} for m in METHODS}
pc = {m: {r["concept"]: r for r in fe.per_concept_table(trees[m], D).iter_rows(named=True)} for m in METHODS}

# sanity check: same semantic fidelity as the final notebook's results table
for m in METHODS:
    fid = float(np.mean([pc[m][c]["fidelity"] for c in M]))
    ref = perf.filter((pl.col("method") == m) & (pl.col("k") == CFG.k))["semantic_fidelity"][0]
    assert np.isclose(fid, ref), (m, fid, ref)
print("tokenized:", METHODS, "-- semantic fidelity matches the final notebook for every method")

tokenized: ['0.9', 'highest_degree_dist_1', 'personalized_pagerank', 'highest_degree', 'closeness_centrality', 'pagerank', 'most_children'] -- semantic fidelity matches the final notebook for every method


## 4–5. Same vs different context trees

Two trees are the **same** when their signatures are equal (same tokens in the same contexts, same uncovered branches; depths ignored, since raters do not see them).

The table splits the concepts by whether the concept is itself a token.

In [8]:
def compare(base):
    rows = []
    for c in M:
        tok_o, tok_b = c in T[OURS], c in T[base]
        rows.append({"concept": c, "same_tree": sig[OURS][c] == sig[base][c],
                     "token_in": "both" if tok_o and tok_b else "ours only" if tok_o else
                                 "baseline only" if tok_b else "neither"})
    return pl.DataFrame(rows)


summary = []
for b in CFG.baselines:
    d = compare(b)
    n_same = int(d["same_tree"].sum())
    summary.append({
        "baseline": b,
        "same tree": n_same,
        "same tree (%)": round(100 * n_same / len(M), 1),
        "  of which concept is a token in both": d.filter(pl.col("same_tree") & (pl.col("token_in") == "both")).height,
        "different tree": len(M) - n_same,
        "  token in ours only": d.filter(~pl.col("same_tree") & (pl.col("token_in") == "ours only")).height,
        "  token in baseline only": d.filter(~pl.col("same_tree") & (pl.col("token_in") == "baseline only")).height,
        "  token in neither": d.filter(~pl.col("same_tree") & (pl.col("token_in") == "neither")).height,
    })
df_summary = pl.DataFrame(summary)
df_summary.write_csv(out(f"same_vs_different_k{CFG.k}.csv"))
df_summary

baseline,same tree,same tree (%),of which concept is a token in both,different tree,token in ours only,token in baseline only,token in neither
str,i64,f64,i64,i64,i64,i64,i64
"""highest_degree_dist_1""",4832,29.5,789,11538,2418,199,8921
"""personalized_pagerank""",6298,38.5,1233,10072,1974,472,7626
"""highest_degree""",3158,19.3,483,13212,2724,174,10314
"""closeness_centrality""",3894,23.8,531,12476,2676,154,9646
"""pagerank""",4141,25.3,486,12229,2721,67,9441
"""most_children""",1808,11.0,636,14562,2571,330,11661


## 6–7. Categories → eligible pool

Every concept gets a category (see the table at the top). The funnel applies the inclusion criteria; the category table gives the sizes used to weight the results.

**Note on the size limit:** in `ours_token` it keeps the baseline's shorter, better decompositions, so it slightly works *against* ours (conservative). In the other categories it barely changes the automatic score split.

In [9]:
def n_tokens(tree):
    return sum(len(ctx.tokens) for ctx in _iter_contexts(tree))


def per_concept_comparison(base):
    rows = []
    for c in M:
        o, b = pc[OURS][c], pc[base][c]
        rows.append({
            "concept": c, "label": label(c), "concept_type": concept_type.get(c),
            "official_status": status.get(c),
            "same_tree": sig[OURS][c] == sig[base][c],
            "token_ours": c in T[OURS], "token_baseline": c in T[base],
            "n_def_relations": len(adj.get(c, ())),
            "ours_n_tokens": o["n_tokens"], "base_n_tokens": b["n_tokens"],
            "ours_n_uncovered": o["n_uncovered"], "base_n_uncovered": b["n_uncovered"],
            "ours_fidelity": o["fidelity"], "base_fidelity": b["fidelity"],
        })
    d = pl.DataFrame(rows).with_columns((pl.col("ours_n_tokens") + pl.col("base_n_tokens")).alias("n_tokens_shown"))
    category = (pl.when(pl.col("token_ours") & pl.col("token_baseline")).then(pl.lit("both_token"))
                  .when(pl.col("token_ours")).then(pl.lit("ours_token"))
                  .when(pl.col("token_baseline")).then(pl.lit("baseline_token"))
                  .when(pl.col("same_tree")).then(pl.lit("same_tree"))
                  .otherwise(pl.lit("both_decompose")))
    d = d.with_columns(category.alias("category"))
    return d.with_columns(
        (pl.col("official_status").is_not_null() & ~pl.col("same_tree")
         & (pl.col("n_tokens_shown") <= CFG.max_tokens_shown)).alias("eligible"))


CATEGORY_ORDER = ["both_token", "same_tree", "ours_token", "baseline_token", "both_decompose"]


def allocate(total, sizes):
    # split `total` over strata proportionally to `sizes` (largest remainder)
    tot = sum(sizes.values())
    raw = {k: total * v / tot for k, v in sizes.items()}
    out_ = {k: int(np.floor(x)) for k, x in raw.items()}
    for k in sorted(raw, key=lambda k: raw[k] - out_[k], reverse=True)[:total - sum(out_.values())]:
        out_[k] += 1
    return out_


comp, pools, funnel, cat_tables, ALLOC = {}, {}, [], {}, {}
for b in COMPARE:
    d = comp[b] = per_concept_comparison(b)
    d.write_parquet(out(f"per_concept_comparison_{b}_k{CFG.k}.parquet"))
    s1 = d.filter(pl.col("official_status").is_not_null())
    s2 = s1.filter(~pl.col("same_tree"))
    s3 = s2.filter(pl.col("n_tokens_shown") <= CFG.max_tokens_shown)
    assert s3.height == d["eligible"].sum()
    assert set(s3["category"].unique().to_list()) <= set(RATED)
    pools[b] = {(c, st): sorted(s3.filter((pl.col("category") == c) & (pl.col("official_status") == st))["concept"].to_list())
                for c in RATED for st in STATUSES}
    # sample size per (category, status): same status ratio as the population of the category
    pop = {(c, st): s1.filter((pl.col("category") == c) & (pl.col("official_status") == st)).height
           for c in CATEGORY_ORDER for st in STATUSES}
    ALLOC[b] = {}
    for c in RATED:
        ALLOC[b].update({(c, st): n for st, n in allocate(N_RATED[c], {st: pop[(c, st)] for st in STATUSES}).items()})
    for step, x in [("1. HERO concepts M", d), ("2. in the official release (fully defined or primitive)", s1),
                    ("3. trees differ (identical trees: counted as no difference)", s2),
                    (f"4. at most {CFG.max_tokens_shown} tokens in A + B  -> eligible pool", s3)]:
        funnel.append({"baseline": b, "criterion": step, "concepts": x.height})
    cat_tables[b] = pl.DataFrame([{
        "baseline": b, "category": c, "status": st, "rated": c in RATED,
        "population (weight)": pop[(c, st)],
        "eligible (size limit)": len(pools[b][(c, st)]) if c in RATED else None,
        "sampled": ALLOC[b].get((c, st), 0)} for c in CATEGORY_ORDER for st in STATUSES])
    cat_tables[b].write_csv(out(f"strata_{b}_k{CFG.k}.csv"))

df_funnel = pl.DataFrame(funnel).pivot(on="baseline", index="criterion", values="concepts")
df_funnel.write_csv(out(f"funnel_k{CFG.k}.csv"))
df_funnel

criterion,personalized_pagerank
str,i64
"""1. HERO concepts M""",16370
"""2. in the official release (fully defined or primitive)""",11796
"""3. trees differ (identical trees: counted as no difference)""",7490
"""4. at most 16 tokens in A + B -> eligible pool""",3726


In [10]:
# what the pool looks like: semantic tags, tree sizes, definition sizes (rater workload)
def describe_pool(b):
    d = comp[b].filter(pl.col("eligible"))
    tags = Counter(l.rsplit("(", 1)[-1].rstrip(")") if l.endswith(")") else "?" for l in d["label"].to_list())
    print(f"--- {b}: {d.height:,} eligible concepts")
    print("  semantic tags:", ", ".join(f"{t} {n:,}" for t, n in tags.most_common(6)))
    return d.select(
        pl.lit(b).alias("baseline"),
        pl.col("n_def_relations").median().alias("def relations, median"),
        pl.col("ours_n_tokens").median().alias("ours tokens, median"),
        pl.col("ours_n_tokens").quantile(0.9, "nearest").alias("ours tokens, p90"),
        pl.col("base_n_tokens").median().alias("base tokens, median"),
        pl.col("base_n_tokens").quantile(0.9, "nearest").alias("base tokens, p90"),
        (pl.col("ours_n_uncovered") > 0).mean().alias("ours has ∅ (share)"),
        (pl.col("base_n_uncovered") > 0).mean().alias("base has ∅ (share)"),
        ((pl.col("ours_n_tokens") == 0) | (pl.col("base_n_tokens") == 0)).sum().alias("a side without tokens"),
    )


pl.concat([describe_pool(b) for b in COMPARE])

--- personalized_pagerank: 3,726 eligible concepts
  semantic tags: disorder 1,766, finding 734, substance 557, procedure 362, physical object 78, medicinal product 68


baseline,"def relations, median","ours tokens, median","ours tokens, p90","base tokens, median","base tokens, p90",ours has ∅ (share),base has ∅ (share),a side without tokens
str,f64,f64,f64,f64,f64,f64,f64,u32
"""personalized_pagerank""",3.0,3.0,8.0,4.0,9.0,0.0,0.001342,0


In [11]:
# strata (category x status): population = weight of the final analysis, eligible, sampled
cat_tables[COMPARE[0]]

baseline,category,status,rated,population (weight),eligible (size limit),sampled
str,str,str,bool,i64,i64,i64
"""personalized_pagerank""","""both_token""","""defined""",false,776,null,0
"""personalized_pagerank""","""both_token""","""primitive""",false,457,null,0
"""personalized_pagerank""","""same_tree""","""defined""",false,1062,null,0
"""personalized_pagerank""","""same_tree""","""primitive""",false,2011,null,0
"""personalized_pagerank""","""ours_token""","""defined""",true,1200,648,20
"""personalized_pagerank""","""ours_token""","""primitive""",true,620,558,10
"""personalized_pagerank""","""baseline_token""","""defined""",true,272,259,12
"""personalized_pagerank""","""baseline_token""","""primitive""",true,200,200,8
"""personalized_pagerank""","""both_decompose""","""defined""",true,3535,1023,34


### Automatic score by stratum (for us only)

Per-concept semantic fidelity of ours vs the baseline, split by the categories of the funnel. It shows **where** each method wins on the automatic score, and what criterion 3 (concept is not a token) removes. Raters never see these numbers.

In [12]:
def fidelity_by_stratum(b):
    d = comp[b]
    diff = d.filter(~pl.col("same_tree"))
    neither = diff.filter(~pl.col("token_ours") & ~pl.col("token_baseline"))
    strata = [
        ("all M", d),
        ("different tree", diff),
        ("  concept is a token in ours only", diff.filter(pl.col("token_ours") & ~pl.col("token_baseline"))),
        ("  concept is a token in the baseline only", diff.filter(~pl.col("token_ours") & pl.col("token_baseline"))),
        ("  token in neither", neither),
        ("    primitive", neither.filter(pl.col("official_status") == "primitive")),
        ("    not in the official release", neither.filter(pl.col("official_status").is_null())),
        ("    fully defined", neither.filter(pl.col("official_status") == "defined")),
        (f"      and at most {CFG.max_tokens_shown} tokens in A + B", neither.filter((pl.col("official_status") == "defined") & (pl.col("n_tokens_shown") <= CFG.max_tokens_shown))),
        ("different tree and fully defined (criterion 3 not applied)", diff.filter(pl.col("official_status") == "defined")),
    ]
    return pl.DataFrame([{
        "baseline": b, "stratum": name, "concepts": x.height,
        "ours better (%)": round(100 * float((x["ours_fidelity"] > x["base_fidelity"]).mean()), 1),
        "equal (%)": round(100 * float((x["ours_fidelity"] == x["base_fidelity"]).mean()), 1),
        "baseline better (%)": round(100 * float((x["ours_fidelity"] < x["base_fidelity"]).mean()), 1),
        "mean fidelity ours": round(float(x["ours_fidelity"].mean()), 3),
        "mean fidelity baseline": round(float(x["base_fidelity"].mean()), 3),
        "mean tokens ours": round(float(x["ours_n_tokens"].mean()), 1),
        "mean tokens baseline": round(float(x["base_n_tokens"].mean()), 1),
    } for name, x in strata])


df_strata = pl.concat([fidelity_by_stratum(b) for b in COMPARE])
df_strata.write_csv(out(f"fidelity_by_stratum_k{CFG.k}.csv"))
df_strata.filter(pl.col("baseline") == COMPARE[0])

baseline,stratum,concepts,ours better (%),equal (%),baseline better (%),mean fidelity ours,mean fidelity baseline,mean tokens ours,mean tokens baseline
str,str,i64,f64,f64,f64,f64,f64,f64,f64
"""personalized_pagerank""","""all M""",16370,20.2,38.9,41.0,0.832,0.815,7.5,8.7
"""personalized_pagerank""","""different tree""",10072,32.8,0.6,66.6,0.796,0.767,10.0,12.0
"""personalized_pagerank""",""" concept is a token in ours only""",1974,100.0,0.0,0.0,1.0,0.71,1.0,14.4
"""personalized_pagerank""",""" concept is a token in the baseline only""",472,0.0,0.0,100.0,0.871,1.0,4.3,1.0
"""personalized_pagerank""",""" token in neither""",7626,17.4,0.8,81.8,0.738,0.767,12.7,12.1
"""personalized_pagerank""",""" primitive""",1663,16.0,1.3,82.7,0.767,0.795,9.4,9.2
"""personalized_pagerank""",""" not in the official release""",2428,23.3,0.9,75.7,0.744,0.763,10.7,10.8
"""personalized_pagerank""",""" fully defined""",3535,13.9,0.5,85.6,0.721,0.757,15.5,14.3
"""personalized_pagerank""",""" and at most 16 tokens in A + B""",1023,14.4,0.3,85.3,0.809,0.839,6.0,5.4


## 8. Sample and context trees

For each baseline in `COMPARE` and each stratum (rated category × status):

* the eligible concepts of the stratum are shuffled once with `CFG.seed`;
* the first ones (as many as allocated in Section 6–7) are **sampled**, the rest are **reserves** (used in order if a concept must be replaced; write the reason down);
* every sampled concept gets its official definition and both context trees, saved as JSON for the rater form.

A/B positions are **not** assigned here; that happens when the form is built.

In [13]:
def pretty_rel(r):
    # FINDING_SITE -> Finding site
    return r.replace("_", " ").capitalize()


def concept_json(c):
    return {"id": c, "name": label(c)}


def official_definition(c):
    # direct relationships of c in G_D (edges to the root concept were dropped when G_D was built)
    edges = adj.get(c, ())
    return {
        "parents": [concept_json(v) for r, v in edges if r == "IS_A"],
        "attributes": [{"rel": r, "rel_label": pretty_rel(r), **concept_json(v)} for r, v in edges if r != "IS_A"],
    }


def context_json(ctx):
    return {
        "tokens": [{**concept_json(t), "depth": int(d)} for t, d in ctx.tokens],
        "uncovered": bool(ctx.uncovered),
        "subs": [{"rel": s.relation, "rel_label": pretty_rel(s.relation),
                  "destination": concept_json(s.destination), "context": context_json(s)}
                 for s in ctx.subcontexts],
    }


samples = {}
for b in COMPARE:
    frames = []
    for ci, cat in enumerate(RATED):
        for si, st in enumerate(STATUSES):
            n_take = ALLOC[b][(cat, st)]
            order = np.random.default_rng([CFG.seed, ci, si]).permutation(pools[b][(cat, st)]).tolist()
            assert len(order) >= n_take, f"not enough concepts in {cat} / {st}"
            split = (["main"] * n_take + ["reserve"] * len(order))[:len(order)]
            frames.append(pl.DataFrame({"category": cat, "status": st, "rank": range(1, len(order) + 1), "concept": order,
                                        "label": [label(c) for c in order], "split": split}))
    df_order = pl.concat(frames)
    df_order.write_csv(out(f"pool_order_{b}_k{CFG.k}_seed{CFG.seed}.csv"))

    picked = df_order.filter(pl.col("split") != "reserve")
    records = []
    for r in picked.iter_rows(named=True):
        c = r["concept"]
        records.append({
            "category": r["category"], "status": r["status"], "rank": r["rank"], "split": r["split"], **concept_json(c),
            "definition": official_definition(c),
            "ours": context_json(trees[OURS][c]),
            "baseline": context_json(trees[b][c]),
            "metrics": {"ours": {k_: pc[OURS][c][k_] for k_ in ("n_tokens", "n_uncovered", "fidelity")},
                        "baseline": {k_: pc[b][c][k_] for k_ in ("n_tokens", "n_uncovered", "fidelity")}},
        })
    meta = {"lam": CFG.lam, "k": CFG.k, "D": D, "baseline": b, "seed": CFG.seed,
            "n_rated": N_RATED, "allocation": {f"{c}/{st}": n for (c, st), n in ALLOC[b].items()},
            "pool_size": {f"{c}/{st}": len(v) for (c, st), v in pools[b].items()}}
    with open(out(f"sample_{b}_k{CFG.k}_seed{CFG.seed}.json"), "w", encoding="utf-8") as f:
        json.dump({"meta": meta, "concepts": records}, f, ensure_ascii=False, indent=1)
    samples[b] = picked.join(comp[b].select("concept", "n_def_relations", "ours_n_tokens", "base_n_tokens",
                                            "ours_n_uncovered", "base_n_uncovered", "ours_fidelity", "base_fidelity"),
                             on="concept", how="left")
    for (cat, st), n_take in ALLOC[b].items():
        print(f"{b} / {cat} / {st}: eligible {len(pools[b][(cat, st)]):,} -> sampled {n_take}")

personalized_pagerank / ours_token / defined: eligible 648 -> sampled 20
personalized_pagerank / ours_token / primitive: eligible 558 -> sampled 10
personalized_pagerank / baseline_token / defined: eligible 259 -> sampled 12
personalized_pagerank / baseline_token / primitive: eligible 200 -> sampled 8
personalized_pagerank / both_decompose / defined: eligible 1,023 -> sampled 34
personalized_pagerank / both_decompose / primitive: eligible 1,038 -> sampled 16


In [14]:
# the sampled concepts of the first baseline (fidelity is for us only: never shown to raters)
samples[COMPARE[0]]

category,status,rank,concept,label,split,n_def_relations,ours_n_tokens,base_n_tokens,ours_n_uncovered,base_n_uncovered,ours_fidelity,base_fidelity
str,str,i64,str,str,str,i64,i64,i64,i64,i64,f64,f64
"""ours_token""","""defined""",1,"""402701002""","""Dermatosis of lip (disorder)""","""main""",3,1,9,0,1,1.0,0.47
"""ours_token""","""defined""",2,"""47695004""","""Inability to cope (finding)""","""main""",4,1,6,0,0,1.0,0.8
"""ours_token""","""defined""",3,"""398695002""","""Product containing atenolol and nifedipine (medicinal product)""","""main""",4,1,13,0,0,1.0,0.684615
"""ours_token""","""defined""",4,"""22495007""","""Abnormal weight (finding)""","""main""",3,1,7,0,0,1.0,0.771429
"""ours_token""","""defined""",5,"""58126003""","""Postoperative wound infection (disorder)""","""main""",7,1,15,0,0,1.0,0.793333
"""ours_token""","""defined""",6,"""30270006""","""Infection caused by Human parainfluenza virus 3 (disorder)""","""main""",4,1,4,0,0,1.0,0.7
"""ours_token""","""defined""",7,"""4598005""","""Osteomalacia (disorder)""","""main""",4,1,8,0,0,1.0,0.7375
"""ours_token""","""defined""",8,"""127144009""","""Mesenteric lymphadenopathy (disorder)""","""main""",2,1,7,0,0,1.0,0.642857
"""ours_token""","""defined""",9,"""282447006""","""Injury of nasal septum (disorder)""","""main""",4,1,11,0,0,1.0,0.772727


In [15]:
# workload and score preview of each sample (raters see trees, not these numbers)
pl.concat([
    s.filter(pl.col("split") == "main").group_by("category", "status", maintain_order=True).agg(
        pl.lit(b).alias("baseline"),
        pl.len().alias("concepts"),
        pl.col("ours_n_tokens").median().alias("ours tokens, median"),
        pl.col("base_n_tokens").median().alias("base tokens, median"),
        (pl.col("ours_n_tokens") + pl.col("base_n_tokens")).max().alias("tokens shown, max (A+B)"),
        (pl.col("ours_fidelity") > pl.col("base_fidelity")).sum().alias("fidelity ours > base"),
        (pl.col("ours_fidelity") == pl.col("base_fidelity")).sum().alias("fidelity equal"),
        (pl.col("ours_fidelity") < pl.col("base_fidelity")).sum().alias("fidelity ours < base"),
    ) for b, s in samples.items()])

category,status,baseline,concepts,"ours tokens, median","base tokens, median","tokens shown, max (A+B)",fidelity ours > base,fidelity equal,fidelity ours < base
str,str,str,u32,f64,f64,i64,u32,u32,u32
"""ours_token""","""defined""","""personalized_pagerank""",20,1.0,7.0,16,20,0,0
"""ours_token""","""primitive""","""personalized_pagerank""",10,1.0,2.0,9,10,0,0
"""baseline_token""","""defined""","""personalized_pagerank""",12,5.5,1.0,14,0,0,12
"""baseline_token""","""primitive""","""personalized_pagerank""",8,2.0,1.0,14,0,0,8
"""both_decompose""","""defined""","""personalized_pagerank""",34,5.5,5.5,16,4,0,30
"""both_decompose""","""primitive""","""personalized_pagerank""",16,5.0,5.0,16,2,0,14


In [16]:
# context trees of the sampled concepts: printed here for the pilot, saved to a text file for all
def render_concept(c, base):
    d = official_definition(c)
    lines = ["=" * 100, f"{label(c)} ({c})", "official definition:"]
    lines += [f"    Is a: {p['name']} ({p['id']})" for p in d["parents"]]
    lines += [f"    {a['rel_label']}: {a['name']} ({a['id']})" for a in d["attributes"]]
    for name, m in (("OURS", OURS), (f"BASELINE ({base})", base)):
        r = pc[m][c]
        lines += [f"--- {name}: {r['n_tokens']} tokens, {r['n_uncovered']} uncovered, fidelity {r['fidelity']:.3f}",
                  render_context_tree(trees[m][c], id_to_label)]
    return "\n".join(lines)


for b, s in samples.items():
    text = "\n".join(render_concept(r["concept"], b) + f"\n[{r['category']} | {r['status']} | {r['split']} #{r['rank']}]" for r in s.iter_rows(named=True))
    with open(out(f"sample_trees_{b}_k{CFG.k}_seed{CFG.seed}.txt"), "w", encoding="utf-8") as f:
        f.write(text)

b = COMPARE[0]
for r in samples[b].group_by("category", "status", maintain_order=True).head(1).iter_rows(named=True):   # one example per stratum
    print(render_concept(r["concept"], b))

Dermatosis of lip (disorder) (402701002)
official definition:
    Is a: Disorder of lip (disorder) (90678009)
    Is a: Disorder of skin of head (disorder) (400082007)
    Finding site: Skin structure of lip (body structure) (88089004)
--- OURS: 1 tokens, 0 uncovered, fidelity 1.000
ROOT
└── • Dermatosis of lip (disorder) (402701002)  d=0
--- BASELINE (personalized_pagerank): 9 tokens, 1 uncovered, fidelity 0.470
ROOT
├── • Disorder of lip (disorder) (90678009)  d=1
├── • Disorder of skin of head (disorder) (400082007)  d=1
└── FINDING_SITE -> Skin structure of lip (body structure) (88089004)
    ├── • Structure of skin and/or skin associated mucous membrane of external body orifice (body structure) (1351226008)  d=5
    ├── • Oral soft tissues structure (body structure) (74336005)  d=5
    ├── • Structure of region of face (body structure) (123853000)  d=5
    ├── • Mouth region part (body structure) (127945006)  d=5
    ├── • Skin structure of head (body structure) (70762009)  d=9
  

## 9. One rater form for every expert, and one ground-truth file

**One** offline HTML form is sent to **every** expert. Each expert types their initials when they start; answers are saved in the browser, and "Download my answers" gives a CSV named with their initials.

* **Concept order:** different for each expert, derived from their initials (the same initials always give the same order). The CSV always lists the concepts in the common item order, so the files of several experts line up row by row.
* **A/B positions:** random per concept, balanced within each stratum (category × status), the same for every expert.
* **A concept kept as a token** is shown as the concept itself ("Same concept"). Both methods have such cases, so it does not reveal which side is ours.
* **Bold:** a concept that is not at the same place (same relation path) in the other version.
* **Dark green relation and concept:** the same relation to the same concept as in the official definition (direct relations only); both the relation and the concept are dark green.
* **Status:** each concept shows whether it is fully defined or primitive. For primitive concepts the raters also use the concept name, and the "definition not sufficient" box is hidden (it is true by construction).
* **Blinding:** the form contains only the concepts and the two trees. No method name, no score. Which side is ours is only in the ground-truth file, which stays with you.
* `expert_eval/unblind.py` merges every expert's CSV with the ground truth and compares the experts.

In [17]:
FORM_TEMPLATE = "expert_eval/rating_form_template.html"   # in this repo
FORM_LANG = "en"                                          # default language of the form ("fr" possible)
FORM_VIEW = "graph"                                       # default view ("tree" possible)
RETURN_TO = "Yuanyuan Zheng"
FORM_VERSION = "3.0"
FORM_FILE = f"expert_form_k{CFG.k}.html"
TRUTH_FILE = f"GROUND_TRUTH_do_not_send_k{CFG.k}.csv"

with open(FORM_TEMPLATE, encoding="utf-8") as f:
    template = f.read()
assert template.count("__STUDY_DATA__") == 1


def placed_items(cj, path=()):
    # every (relation path, token) and (relation path, ∅) of a tree; the root path () is "Is a"
    items = {(path, t_["id"]) for t_ in cj["tokens"]}
    if cj["uncovered"]:
        items.add((path, "∅"))
    for s_ in cj["subs"]:
        items |= placed_items(s_["context"], path + (s_["rel_label"],))
    return items


def form_context(cj, other, cid, official, path=(), def_ids=None):
    # what a rater sees of one context: tokens, uncovered IS-A branch, attribute branches (no depth, no destination)
    #   diff: the token (or ∅) is not at the same relation path in the other version   -> bold
    #   self: the token is the concept itself (concept kept as a token)                 -> "Same concept"
    #   def : (relation, token) is a direct relationship of the official definition     -> relation AND token dark green
    parents, attrs = official
    if def_ids is None:                                   # root: tokens reached by IS-A, official parents match
        def_ids = parents if not path else set()
    subs = []
    for s_ in cj["subs"]:
        # tokens of a direct attribute branch that are exactly the official value of that attribute
        sub_def = {t2["id"] for t2 in s_["context"]["tokens"] if (s_["rel"], t2["id"]) in attrs} if not path else set()
        subs.append({"rel": s_["rel_label"], **({"def": 1} if sub_def else {}),
                     "context": form_context(s_["context"], other, cid, official, path + (s_["rel_label"],), sub_def)})
    out_ = {"tokens": [{"id": t_["id"], "name": t_["name"],
                        **({"diff": 1} if (path, t_["id"]) not in other else {}),
                        **({"self": 1} if (not path and t_["id"] == cid) else {}),
                        **({"def": 1} if t_["id"] in def_ids else {})}
                       for t_ in cj["tokens"]],
            "uncovered": cj["uncovered"],
            "subs": subs}
    if cj["uncovered"] and (path, "∅") not in other:
        out_["udiff"] = 1
    return out_


def form_definition(dj):
    attrs = [{"rel": a["rel_label"], "id": a["id"], "name": a["name"]} for a in dj["attributes"]]
    return {"parents": dj["parents"], "groups": [attrs] if attrs else []}


b = COMPARE[0]
with open(out(f"sample_{b}_k{CFG.k}_seed{CFG.seed}.json"), encoding="utf-8") as f:
    sample = [r for r in json.load(f)["concepts"] if r["split"] == "main"]

# common item order (1..n): one random order fixed by the seed, so item numbers say nothing about the category
sample = [sample[i] for i in np.random.default_rng([CFG.seed, 9]).permutation(len(sample))]

# A/B: balanced within each stratum (category x status), the same for every expert
ours_is_A = {}
for si, (cat, st) in enumerate(sorted({(r["category"], r["status"]) for r in sample})):
    ids = [r["id"] for r in sample if (r["category"], r["status"]) == (cat, st)]
    flags = np.random.default_rng([CFG.seed, 100 + si]).permutation([True] * ((len(ids) + 1) // 2) + [False] * (len(ids) // 2))
    ours_is_A.update(zip(ids, flags))

study_id = f"tokeval-k{CFG.k}"
items, truth = [], []
for item, r in enumerate(sample, 1):
    oa = bool(ours_is_A[r["id"]])
    A, B = (r["ours"], r["baseline"]) if oa else (r["baseline"], r["ours"])
    official = ({p["id"] for p in r["definition"]["parents"]}, {(a["rel"], a["id"]) for a in r["definition"]["attributes"]})
    items.append({"item": item, "concept_id": r["id"], "name": r["name"], "status": r["status"],
                  "definition": form_definition(r["definition"]),
                  "A": form_context(A, placed_items(B), r["id"], official),
                  "B": form_context(B, placed_items(A), r["id"], official)})
    truth.append({"study_id": study_id, "item": item, "concept_id": r["id"], "concept_name": r["name"],
                  "category": r["category"], "status": r["status"],
                  "A": "ours" if oa else "baseline", "B": "baseline" if oa else "ours",
                  "ours_method": f"lambda={CFG.lam}, k={CFG.k}", "baseline_method": f"{b}, k={CFG.k}"})

data = {"study_id": study_id, "form_version": FORM_VERSION, "rater_id": "", "shuffle_by_rater": True,
        "return_to": RETURN_TO, "lang": FORM_LANG, "view_default": FORM_VIEW, "demo": False, "concepts": items}
html = template.replace("__STUDY_DATA__", json.dumps(data, ensure_ascii=False).replace("</", "<\\/"))
low = html.lower()
assert "pagerank" not in low and '"ours"' not in low and "baseline_method" not in low, "blinding leak"
with open(out(FORM_FILE), "w", encoding="utf-8") as f:
    f.write(html)
df_truth = pl.DataFrame(truth)
df_truth.write_csv(out(TRUTH_FILE))
print(f"{FORM_FILE}: {len(items)} concepts (send this file to every expert)")
print(f"{TRUTH_FILE}: {df_truth.height} rows (keep it)")
df_truth.group_by("category", "status", "A").len().sort("category", "status", "A")

expert_form_k5000.html: 100 concepts (send this file to every expert)
GROUND_TRUTH_do_not_send_k5000.csv: 100 rows (keep it)


category,status,A,len
str,str,str,u32
"""baseline_token""","""defined""","""baseline""",6
"""baseline_token""","""defined""","""ours""",6
"""baseline_token""","""primitive""","""baseline""",4
"""baseline_token""","""primitive""","""ours""",4
"""both_decompose""","""defined""","""baseline""",17
"""both_decompose""","""defined""","""ours""",17
"""both_decompose""","""primitive""","""baseline""",8
"""both_decompose""","""primitive""","""ours""",8
"""ours_token""","""defined""","""baseline""",10


## Files written to `CFG.out_dir`

| File | Content |
|---|---|
| `same_vs_different_k5000.csv` | steps 4–5, every baseline |
| `funnel_k5000.csv` | steps 6–7, one column per compared baseline |
| `fidelity_by_stratum_k5000.csv` | automatic score of ours vs baseline per stratum |
| `per_concept_comparison_<baseline>_k5000.parquet` | one row per concept of $M$, with the `eligible` flag |
| `pool_order_<baseline>_k5000_seed2026.csv` | eligible pool in random order: pilot / main / reserve |
| `sample_<baseline>_k5000_seed2026.json` | sampled concepts: official definition + both trees (input of the rater form) |
| `sample_trees_<baseline>_k5000_seed2026.txt` | the same trees, printed, for a quick read |
| `expert_form_k5000.html` | the one form for every expert (send this) |
| `GROUND_TRUTH_do_not_send_k5000.csv` | which side is ours, category and status per concept (keep it) |
| `strata_<baseline>_k5000.csv` | strata sizes (weights of the analysis), eligible, sampled |

**After the study:** put every expert's CSV in one folder and run
`python expert_eval/unblind.py GROUND_TRUTH_do_not_send_k5000.csv <folder>`
It writes one long table (one row per expert and concept) and one wide table (one row per concept, one column per expert), and prints the agreement between experts.